##### Import statements:

In [ ]:
import os
import pathlib
import inspect
import functools
import pickle
import numpy as np
import pandas as pd
import json
import socket
import multiprocessing as mp
import functools
import itertools
import torch
import matplotlib.pyplot as plt
import random
import string

hostname = socket.gethostname()

if 'rc.zi.columbia.edu' in hostname:
    from ws.general import find_df_constants, matches_template, class_def2str
    from ws.simulate_task import load_sim_params, load_task_def, simulate_session
    from ws.miscellaneous_sparseauto import mdl_geometry_pipeline, fmt_ae_metadata, generate_hparams_df, causal_mask
    from ws.plot import plot_iterate_autoencoder_results, plot_ccgps_by_layer, plot_pars_by_layer
    base = os.path.join('/', 'mnt', 'smb', 'locker', 'issa-locker', 'users', 'Dan', 'code', 'ws') 
else:
    from simulation_whiskers.general import find_df_constants, matches_template, class_def2str
    from simulation_whiskers.simulate_task import load_sim_params, load_task_def, simulate_session, causal_mask
    from simulation_whiskers.miscellaneous_sparseauto import mdl_geometry_pipeline, fmt_ae_metadata, generate_hparams_df, causal_mask
    #from simulation_whiskers.plot import plot_iterate_autoencoder_results, plot_autoencoder_geometry
    from simulation_whiskers.plot import plot_iterate_autoencoder_results, plot_ccgps_by_layer, plot_pars_by_layer
    base = os.path.join('C:\\', 'Users', 'danie' 'Documents', 'code_libraries', 'simulation_whiskers')

from analysis_metadata.analysis_metadata import Metadata, increment_dir_name, write_metadata
import time

##### Define parameters:

In [ ]:
# Define task definitions:
task_defs = [
    
    # Task 0:
    [
     functools.partial(matches_template, template={'freq_sh' : 2}), 
     functools.partial(matches_template, template={'freq_sh' : 15})
     ],
    
    # Task 1:
    [
     functools.partial(matches_template, template={'time_mov' : 10}),
     functools.partial(matches_template, template={'time_mov' : 17})
     ]
    ]

# Define general variables:
n_repeats = 100 
n_repeats = 2 # < For debugging
n_geo_subsamples = 1
sum_inpt=False
xor=True
zscore_data = False
sig_init = 1.0
save_learning = False
chunked_reconstruction_loss = False

# Entangler simulation parameters:
tngl_concavity = [0]
tngl_n_whisk = 2
tngl_prob_poiss = 1.01
tngl_noise_w = 0.3
tngl_spread = 'auto'
tngl_speed = 2.0
tngl_ini_phase_m = 0
tngl_ini_phase_spr = 100
tngl_delay_time = 0
tngl_freq_m = 3.0
tngl_freq_std = 0.1
tngl_std_reset = 0
tngl_t_total = 2
tngl_dt = 0.1
tngl_dx = 0.01
#n_trials_pre = 200
tngl_n_trials_pre = 50 # < For debugging
tngl_amp = 2
tngl_freq_sh = [2, 15]
tngl_z1 = [4]
tngl_max_rad = 50
tngl_n_rad = 4
tngl_disp = 4.5
tngl_theta = [0]
tngl_steps_mov = [10, 17]
tngl_rad_vec = [6]
tngl_init_position = 0

# Entangler model parameters:
tngl_par_tgt = 0.0
tngl_par_tolerance = 0.01 
tngl_n_hidden = 80
tngl_beta_rec = 1.0
tngl_beta_pr = 100
tngl_beta_sp = 0
tngl_p_norm = 1
#tngl_n_epochs = 200
tngl_n_epochs = 50 # < For debugging
tngl_batch_size = 64
tngl_lr = 0.001
tngl_sig_init = 1 
tngl_sig_neu = 0.1

# Prediction simulation parameters:
concavity = [0]
n_whisk = 2
prob_poiss = 1.01
noise_w = 0.3
spread = 'auto'
speed = 2.0
ini_phase_m = 0
ini_phase_spr = 100
delay_time = 0
freq_m = 3.0
freq_std = 0.1
std_reset = 0
t_total = 2
dt = 0.1
dx = 0.01
#n_trials_pre = 200
n_trials_pre = 50 # < For debugging
amp = 2
freq_sh = [2, 15]
z1 = [4]
max_rad = 50
n_rad = 4
disp = 4.5
theta = [0]
steps_mov = [10, 17]
rad_vec = [6]
init_position = 0

# Prediction model parameters:
mdl_type = 'prediction'
n_hidden = 40
sig_init = 1 
sig_neu = 0.1 
lr = 0.001
beta0 = 0
beta1 = 0
beta_rec = 1
beta_xor = 0
#n_epochs = 200
n_epochs = 50 # < For debugging 
batch_size = 10
beta_sp = 0
beta_pr = 0
p_norm = 2
n_splits = 5
n_predictor_bins = 10
n_predicted_bins = 4
n_offsets = 1

# Compute parameters:
gpu = False
n_cores = 1

# Do some custom, ad-hoc hyperparameter selection:
#beta_lins=10**np.arange(0, 5, 1)
beta_lins= [1]
#beta_lins = np.array([0] + list(beta_lins))
#beta_lins = [0, 10**2.5, 10**5]
n_hiddens = [40]
#n_hiddens = [40, 80, 160]
hparams = [{'beta_rec':x[0], 'n_hidden':x[1]} for x in list(itertools.product(beta_lins, n_hiddens))]
#hparams = None

# Output directory:
if 'rc.zi.columbia' in hostname:
    base_output_directory = os.path.join(base, 'results')
else:
    base_output_directory='E:\\simulation_whiskers\\results\\'
run_base_name='run'
sv=True

##### Generate dataframe of hyperparameters for main model:

In [ ]:
print('Generating dataframe of parameters...')
default_hparams = {'task_defs' : task_defs, 'xor' : xor, 'n_geo_subsamples' : n_geo_subsamples, 
    'zscore_data' : zscore_data, 'save_perf' : False, 'sum_inpt' : sum_inpt, 'chunked_reconstruction_loss' : False, 
    'save_learning' : save_learning, 'gpu' : gpu, 'save_sessions' : False, 'verbose' : False, 'concavity' : concavity, 
    'n_whisk' : n_whisk, 'prob_poiss' : prob_poiss, 'noise_w' : noise_w, 'spread' : spread, 'speed' : speed, 'ini_phase_m' : ini_phase_m, 
    'ini_phase_spr' : ini_phase_spr, 'delay_time' : delay_time, 'freq_m' : freq_m, 'freq_std' : freq_std, 'std_reset' : std_reset, 
    't_total' : t_total, 'dt' : dt, 'dx' : dx, 'n_trials_pre' : n_trials_pre, 'n_repeats' : n_repeats, 'amp' : amp, 'freq_sh' : freq_sh, 
    'z1' : z1, 'max_rad' : max_rad, 'n_rad': n_rad, 'disp' : disp, 'theta' : theta, 'steps_mov' : steps_mov, 'rad_vec' : rad_vec, 
    'init_position' : init_position, 'mdl_type' : mdl_type, 'n_hidden' : n_hidden, 'sig_init' : sig_init, 'sig_neu' : sig_neu, 
    'lr' : lr, 'beta0' : beta0, 'beta1' : beta1, 'beta_rec' : beta_rec, 'beta_xor' : beta_xor, 'beta_sp' : beta_sp, 'beta_pr' : beta_pr, 
    'n_epochs' : n_epochs, 'batch_size' : batch_size, 'p_norm' : p_norm, 'n_splits' : n_splits, 'n_predictor_bins' : n_predictor_bins, 
    'n_predicted_bins' : n_predicted_bins, 'n_offsets' : n_offsets} 
hparams_df = generate_hparams_df(defaults=default_hparams, hparams_manual=hparams)

# Verify parameters before executing:
hparam_strs = list(hparams_df.apply(lambda x : 'model={}, n_hidden={}, beta_rec={}, beta_sp={}, beta_pr={}, n_epochs={}'.format(x.mdl_type,x.n_hidden, x.beta_rec, x.beta_sp, x.beta_pr, x.n_epochs), axis=1))
print('Running following hyperparameters:\n')
print('\n'.join(hparam_strs))
if '__file__' not in dir():
    yn = input('\nProceed? (y/n)')
    if yn == 'y':
        pass
    else: 
        raise AssertionError('User aborted execution.')

##### Deal with some preliminaries:

In [ ]:
simulation_cols = ['concavity', 'n_whisk', 'prob_poiss', 'noise_w', 'spread',
     'speed', 'ini_phase_m', 'ini_phase_spr', 'delay_time', 'freq_m', 'freq_std',
     'std_reset', 't_total', 'dt', 'dx', 'n_trials_pre', 'n_files', 'amp', 'freq_sh',
     'z1', 'max_rad', 'n_rad', 'disp', 'theta', 'steps_mov', 'rad_vec', 'init_position']

autoencoder_cols = ['mdl_type', 'n_hidden', 'sig_init', 'sig_neu', 'lr', 'beta0',
    'beta1', 'beta_rec', 'beta_xor', 'n_epochs', 'batch_size', 'beta_sp', 'p_norm',
    'beta_pr', 'n_splits', 'n_predictor_bins', 'n_predicted_bins', 'n_offsets']

# TODO: A lot of this seems really inefficient; streamline this somehow?

# Put entangler simulation params into dict:
tngl_sim_params = {
    'concavity' : tngl_concavity,
    'n_whisk' : tngl_n_whisk,
    'prob_poiss' : tngl_prob_poiss,
    'noise_w' : tngl_noise_w,
    'spread' : tngl_spread,
    'speed' : tngl_speed,
    'ini_phase_m' : tngl_ini_phase_m,
    'ini_phase_spr' : tngl_ini_phase_spr, 
    'delay_time' : tngl_delay_time, 
    'freq_m' : tngl_freq_m, 
    'freq_std' : tngl_freq_std,
    'std_reset' : tngl_std_reset,
    't_total' : tngl_t_total,
    'dt' : tngl_dt,
    'dx' : tngl_dx,
    'n_trials_pre' : tngl_n_trials_pre, 
    'amp' : tngl_amp,
    'freq_sh' : tngl_freq_sh,
    'z1' : tngl_z1,
    'max_rad' : tngl_max_rad,
    'n_rad' : tngl_n_rad,
    'disp' : tngl_disp,
    'theta' : tngl_theta,
    'steps_mov' : tngl_steps_mov,
    'rad_vec' : tngl_rad_vec,
    'init_position' : tngl_init_position,
}

# Put prediction simulation params into dict:
pred_sim_params = {
    'concavity' : concavity,
    'n_whisk' : n_whisk,
    'prob_poiss' : prob_poiss,
    'noise_w' : noise_w,
    'spread' : spread,
    'speed' : speed,
    'ini_phase_m' : ini_phase_m,
    'ini_phase_spr' : ini_phase_spr, 
    'delay_time' : delay_time, 
    'freq_m' : freq_m, 
    'freq_std' : freq_std,
    'std_reset' : std_reset,
    't_total' : t_total,
    'dt' : dt,
    'dx' : dx,
    'n_trials_pre' : n_trials_pre, 
    'amp' : amp,
    'freq_sh' : freq_sh,
    'z1' : z1,
    'max_rad' : max_rad,
    'n_rad' : n_rad,
    'disp' : disp,
    'theta' : theta,
    'steps_mov' : steps_mov,
    'rad_vec' : rad_vec,
    'init_position' : init_position,
}

n_feat = n_whisk*2

##### Simulate initial whisker data used to fit entangler model:

In [ ]:
## Simulate train and test sessions: 
tngl_train_session=simulate_session(tngl_sim_params, sum_bins=False)
tngl_train_session['split'] = 'train'
tngl_train_session['trial_num'] = np.arange(tngl_train_session.shape[0])

tngl_test_session=simulate_session(tngl_sim_params, sum_bins=False)
tngl_test_session['split'] = 'test'
tngl_test_session['trial_num'] = np.arange(tngl_test_session.shape[0])

# Merge train and test:
tngl_sim_df = pd.concat([tngl_train_session, tngl_test_session], axis=0)

# Unrwap features:
tngl_sim_df['features'] = tngl_sim_df.apply(lambda x : np.reshape(x.features,-1), axis=1)

# Split into predicted and predictor features:
tngl_sim_df = causal_mask(tngl_sim_df, n_feat, n_predictor_bins, n_predicted_bins, n_offsets)
tngl_sim_df['predicted_features'] = tngl_sim_df['predictor_features'] # Predicted and predictor features will be same for entangling autoencoder

##### Fit entangler model:

In [ ]:
from importlib import reload
import ws.miscellaneous_sparseauto
reload(ws.miscellaneous_sparseauto)
from ws.miscellaneous_sparseauto import mdl_geometry_pipeline

In [ ]:
tngl_mdl_id = ''.join(random.choices(string.ascii_letters+string.digits, k=10)) # Generate unique ID for entangler model instance; will be useful for matching model with downstream results 
tngl_mdl_params = {
    'mdl_type' : "autoencoder",
    'n_hidden' : tngl_n_hidden,
    'beta_rec' : tngl_beta_rec,
    'beta_pr' : tngl_beta_pr,    
    'beta_sp' : tngl_beta_sp,    
    'n_epochs' : tngl_n_epochs, 
    'batch_size' : tngl_batch_size,
    'lr' : tngl_lr, 
    'p_norm' : tngl_p_norm, 
    'sig_init' : tngl_sig_init,
    'sig_neu' :  tngl_sig_neu,
    'beta0' : 0,
    'beta1' : 0,
    'beta_xor' : 0,
    'tngl_mdl_id' : tngl_mdl_id 
}

# Repeatedly fit entangler model until parallelism is within desired tolerance:
tngl_par = np.inf
while np.abs(tngl_par - tngl_par_tgt) > tngl_par_tolerance:
    tngl_results = mdl_geometry_pipeline(sim_params, task_defs, tngl_mdl_params, sessions_in=tngl_sim_df, save_learning=False, sum_inpt=False)
    tngl_geo_results = tngl_results['geo_df']
    tngl_geo_results_hidden = tngl_geo_results[np.array(tngl_geo_results.layer=='hidden') & np.array(tngl_geo_results.epoch==np.max(tngl_geo_results.epoch))]
    tngl_par = np.mean(tngl_geo_results_hidden.parallelism)
    print('tngl_par - tngl_par_tgt = {}'.format(tngl_par - tngl_par_tgt))
    
# Extract geometry results, add metadata:
tngl_meta = pd.concat([pd.DataFrame(pd.Series(tngl_mdl_params)).T] * tngl_geo_results.shape[0], axis=0)
tngl_meta.index = np.arange(tngl_meta.shape[0])
tngl_geo_results = pd.concat([tngl_geo_results, tngl_meta], axis=1)

# Extract classifier performance results, add metadata:
tngl_perf_results = tngl_results['perf_df']
tngl_meta = pd.concat([pd.DataFrame(pd.Series(tngl_mdl_params)).T] * tngl_perf_results.shape[0], axis=0)
tngl_meta.index = np.arange(tngl_meta.shape[0])
tngl_perf_results = pd.concat([tngl_perf_results, tngl_meta], axis=1)

# Extract autoencoder representations, add metadata:
if tngl_results['ae_df'] is not None:            
    tngl_ae_results = tngl_results['ae_df']
    tngl_meta = pd.concat([pd.DataFrame(pd.Series(tngl_mdl_params)).T] * tngl_ae_results.shape[0], axis=0)
    tngl_meta.index = np.arange(tngl_meta.shape[0])
    tngl_ae_results = pd.concat([tngl_ae_results, tngl_meta], axis=1)

entangler = tngl_results['mdl']

##### Generate simulated trials for each set of hyperparameters:

In [ ]:
# Generate simulated contacts:
sim_df = pd.DataFrame()
for hidx, hparams_row in hparams_df.iterrows():

    # Test:
    curr_train_sim = simulate_session(pred_sim_params, sum_bins=False)
    curr_train_sim['split'] = 'train'
    curr_train_sim['trial_num'] = np.arange(curr_train_sim.shape[0])

    # Train:
    curr_test_sim = simulate_session(pred_sim_params, sum_bins=False)
    curr_test_sim['split'] = 'test'
    curr_test_sim['trial_num'] = np.arange(curr_train_sim.shape[0])

    # Merge:
    curr_sim = pd.concat([curr_train_sim, curr_test_sim], axis=0)

    # Unrwap features:
    curr_sim['features'] = curr_sim.apply(lambda x : np.reshape(x.features,-1), axis=1)
    
    # Split simulated whisker data into predicted and predictor features:
    curr_sim = causal_mask(curr_sim, n_feat, n_predictor_bins, n_predicted_bins, n_offsets)
    curr_sim['sim_idx'] = hidx
    
    # Concatenate across repeats:
    sim_df = pd.concat([sim_df, curr_sim], axis=0)

# Pass predictor features through entangler model:
sim_df['predictor_features'] = sim_df.apply(lambda x : entangler.enc(torch.Tensor(x.predictor_features.astype(np.float32))).detach().numpy(), axis=1)

##### Iterate over hyperparameters, fit models, analyze geometry on each:

In [ ]:
# Iterate over dicts of hyperparamter combos:
pred_geo_results = pd.DataFrame()
pred_perf_results = pd.DataFrame()
pred_ae_results = pd.DataFrame()
start_mdl = time.time()
for hidx, curr_hparams in hparams_df.iterrows():
    
    # Get current model hypermarameter set:
    curr_autoencoder_params = dict(curr_hparams[autoencoder_cols])

    # Get current simulated whisker data:
    curr_whisker_sim = sim_df[sim_df.sim_idx==hidx]
    print(curr_whisker_sim.shape[0])
    
    # Fit model, test geometry:
    pred_mdl_id = ''.join(random.choices(string.ascii_letters+string.digits, k=10)) # Generate unique ID for prediction model instance
    curr_results=mdl_geometry_pipeline(sim_params,  
        tasks=curr_hparams.task_defs, autoencoder_params=curr_autoencoder_params, xor=curr_hparams.xor, 
        n_geo_subsamples=curr_hparams.n_geo_subsamples, zscore_data=curr_hparams.zscore_data, 
        save_perf=False, sum_inpt=curr_hparams.sum_inpt, chunked_reconstruction_loss=curr_hparams.chunked_reconstruction_loss, 
        sessions_in=curr_whisker_sim, save_learning=curr_hparams.save_learning, gpu=curr_hparams.gpu, save_sessions=False, 
        verbose=True)

    curr_hparams_df = pd.DataFrame(hparams_df.iloc[0]).T

    # Extract geometry results, add metadata:
    curr_geo_results = curr_results['geo_df']
    curr_geo_results['pred_mdl_id'] = pred_mdl_id 
    geo_meta_cols = list(set(curr_hparams_df.columns) - set(curr_geo_results.columns))
    geo_meta = pd.concat([curr_hparams_df[geo_meta_cols]]*curr_geo_results.shape[0],axis=0)
    geo_meta.index = np.arange(geo_meta.shape[0])
    curr_geo_results = pd.concat([curr_geo_results, geo_meta], axis=1)
    pred_geo_results = pd.concat([pred_geo_results, curr_geo_results], axis=0)
    
    # Extract classifier performance results, add metadata:
    curr_perf_results = curr_results['perf_df']
    curr_perf_results['pred_mdl_id'] = pred_mdl_id
    perf_meta_cols = list(set(curr_hparams_df.columns) - set(curr_perf_results.columns))
    perf_meta = pd.concat([curr_hparams_df[perf_meta_cols]]*curr_perf_results.shape[0],axis=0)
    perf_meta.index = np.arange(perf_meta.shape[0])
    curr_perf_results = pd.concat([curr_perf_results, perf_meta], axis=1)
    pred_perf_results = pd.concat([pred_perf_results, curr_perf_results], axis=0)

    # Extract autoencoder representations, add metadata:
    if curr_results['ae_df'] is not None:            
        curr_ae_results = curr_results['ae_df']
        curr_ae_results['pred_mdl_id'] = pred_mdl_id
        ae_meta_cols = list(set(curr_hparams_df.columns) - set(curr_ae_results.columns))
        ae_meta = pd.concat([curr_hparams_df[ae_meta_cols]]*curr_ae_results.shape[0],axis=0)
        ae_meta.index = np.arange(ae_meta.shape[0])
        curr_ae_results = pd.concat([curr_ae_results, ae_meta], axis=1)
        pred_ae_results = pd.concat([pred_ae_results, curr_ae_results])

pred_geo_results['train_partition'] = pred_geo_results.apply(lambda x :str(x.train_partition), axis=1)
stop_mdl = time.time()

##### Aggregate results:

In [ ]:
# Add upstream entangler model ID to prediction model results:  
pred_geo_results['tngl_mdl_id'] = tngl_mdl_id
pred_perf_results['tngl_mdl_id'] = tngl_mdl_id
pred_ae_results['tngl_mdl_id'] = tngl_mdl_id

# Concatenate autoencoder and prediction model results:
geo_results = pd.concat([tngl_geo_results, pred_geo_results], axis=0)
perf_results = pd.concat([tngl_perf_results, pred_perf_results], axis=0)
ae_results = pd.concat([tngl_ae_results, pred_ae_results], axis=0)

# Save to dict:
all_results = dict()
all_results['geo_df'] = geo_results
all_results['perf_df'] = perf_results
all_results['ae_df'] = ae_results

##### Save output:

In [ ]:
if sv:
    
    # Save results dataframe:
    curr_output_directory=increment_dir_name(base_output_directory, run_base_name)
    if not os.path.exists(curr_output_directory):
        pathlib.Path(curr_output_directory).mkdir(parents=True, exist_ok=True)
    results_path = os.path.join(curr_output_directory, 'tngl_pred.pickle')
    pickle.dump(all_results, open(results_path, 'wb'))
    
    M = Metadata()
    metadata_consts = find_df_constants(hparams_df)

    # Write task definitions:
    if 'task_defs' in metadata_consts:
        for t, task in enumerate(metadata_consts['task_defs']):
            curr_task_str = ' vs '.join([class_def2str(x) for x in task])
            M.add_param('task{}'.format(t), curr_task_str)

    # Write entangler simulation parameters:
    M.add_param('tngl_sim_params', tngl_sim_params)
    
    # Write prediction simulation parameters:
    pred_sim_params_out = dict()
    for s in simulation_cols:
        if s in metadata_consts:
            pred_sim_params_out[s] = metadata_consts[s]
    M.add_param('pred_sim_params', pred_sim_params_out)

    # Write entangler model parameters:
    M.add_param('tngl_model_params', tngl_mdl_params)
    
    # Write prediction model parameters:
    prediction_model_params = dict()
    for a in autoencoder_cols:
        if a in metadata_consts:
            prediction_model_params[a] = metadata_consts[a]
    if 'n_offsets' in prediction_model_params and prediction_model_params['n_offsets'] is None:
        prediction_model_params['n_offsets'] = 'auto'
    M.add_param('prediction_model_params', prediction_model_params)

    M.add_output(results_path)
    M.duration = stop_mdl - start_mdl
    metadata_path = os.path.join(curr_output_directory, 'tngl_pred_metadata.json')
    write_metadata(M, metadata_path)